# Stats 426 Final Project
## Pneumonia Detection from Chest X-Rays

Binary classification: **NORMAL vs PNEUMONIA**

**Models:** Logistic Regression, XGBoost, Custom CNN  
**Split:** Stratified 70 / 15 / 15 (train / val / test)

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import matplotlib.pyplot as plt
from sklearn.metrics import (
    ConfusionMatrixDisplay, classification_report,
    roc_auc_score, roc_curve, f1_score, accuracy_score
)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = 'chest_xray/chest_xray'
IMG_SIZE = 64
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

### Load and Pool All Data

In [ ]:
def load_images_flat(base_dir, img_size=64):
    label_map = {'NORMAL': 0, 'PNEUMONIA': 1}
    X, y, paths = [], [], []
    for label_name, label_val in label_map.items():
        folder = os.path.join(base_dir, label_name)
        files = [f for f in sorted(os.listdir(folder)) if not f.startswith('.')]
        for fname in tqdm(files, desc=label_name):
            fpath = os.path.join(folder, fname)
            try:
                img = Image.open(fpath).convert('L').resize((img_size, img_size))
                X.append(np.array(img).flatten().astype(np.float32) / 255.0)
                y.append(label_val)
                paths.append(fpath)
            except Exception as e:
                print(f'Skipping {fpath}: {e}')
    return np.array(X), np.array(y), paths

all_X, all_y, all_paths = [], [], []
for split in ['train', 'test', 'val']:
    split_dir = os.path.join(DATA_DIR, split)
    print(f'Loading {split}...')
    X_s, y_s, p_s = load_images_flat(split_dir)
    all_X.append(X_s)
    all_y.append(y_s)
    all_paths.extend(p_s)

X_all = np.vstack(all_X)
y_all = np.concatenate(all_y)
all_paths = np.array(all_paths)

print(f'\nTotal samples: {len(y_all)}')
print(f'NORMAL (0): {(y_all == 0).sum()}')
print(f'PNEUMONIA (1): {(y_all == 1).sum()}')
print(f'Feature dims: {X_all.shape[1]}')

### Stratified 70 / 15 / 15 Split

In [ ]:
X_train, X_temp, y_train, y_temp, p_train, p_temp = train_test_split(
    X_all, y_all, all_paths, test_size=0.30, random_state=SEED, stratify=y_all
)
X_val, X_test, y_val, y_test, p_val, p_test = train_test_split(
    X_temp, y_temp, p_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print('Split sizes (NORMAL / PNEUMONIA / Total):')
for name, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    n, p = int((y == 0).sum()), int((y == 1).sum())
    print(f'  {name:5s}: {n:4d} / {p:4d} / {len(y):4d}  ({n/len(y)*100:.1f}% NORMAL)')

### EDA

In [ ]:
brightness = X_train.mean(axis=1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(brightness, bins=50, color='steelblue', edgecolor='white')
ax.set_xlabel('Mean Pixel Intensity')
ax.set_ylabel('Count')
ax.set_title('Per-Image Brightness Distribution (Training Set)')
plt.tight_layout()
plt.show()

In [ ]:
rng = np.random.RandomState(SEED)
idxs = rng.choice(len(X_train), 12, replace=False)
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, idx in zip(axes.flat, idxs):
    ax.imshow(X_train[idx].reshape(IMG_SIZE, IMG_SIZE), cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
plt.suptitle('Random Training Samples', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
mean_img = X_train.mean(axis=0).reshape(IMG_SIZE, IMG_SIZE)
std_img = X_train.std(axis=0).reshape(IMG_SIZE, IMG_SIZE)
axes[0].imshow(mean_img, cmap='gray')
axes[0].set_title('Mean Image (All Training)')
axes[0].axis('off')
im = axes[1].imshow(std_img, cmap='hot')
axes[1].set_title('Pixel Std Dev (All Training)')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
contrast = X_train.std(axis=1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(contrast, bins=50, color='tomato', edgecolor='white')
ax.set_xlabel('Per-Image Pixel Std Dev (Contrast)')
ax.set_ylabel('Count')
ax.set_title('Per-Image Contrast Distribution (Training Set)')
plt.tight_layout()
plt.show()

In [ ]:
brightness = X_train.mean(axis=1)
contrast = X_train.std(axis=1)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(brightness, contrast, alpha=0.15, s=6, color='steelblue')
ax.set_xlabel('Mean Pixel Intensity (Brightness)')
ax.set_ylabel('Pixel Std Dev (Contrast)')
ax.set_title('Brightness vs Contrast — Training Images')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.decomposition import PCA

pca_full = PCA(random_state=SEED).fit(X_train)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n90 = np.searchsorted(cumvar, 0.90) + 1
n95 = np.searchsorted(cumvar, 0.95) + 1

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(cumvar) + 1), cumvar, color='steelblue')
ax.axhline(0.90, color='gray', linestyle='--')
ax.axhline(0.95, color='gray', linestyle=':')
ax.axvline(n90, color='tomato', linestyle='--', label=f'90% var = {n90} components')
ax.axvline(n95, color='purple', linestyle=':', label=f'95% var = {n95} components')
ax.set_xlabel('Number of PCA Components')
ax.set_ylabel('Cumulative Explained Variance')
ax.set_title('PCA Cumulative Explained Variance')
ax.legend()
plt.tight_layout()
plt.show()

---
## Model 1: Logistic Regression

---
## Class Balancing

In [ ]:
class_counts = np.bincount(y_train)
imbalance_ratio = class_counts[1] / class_counts[0]
print(f'Train — NORMAL: {class_counts[0]}, PNEUMONIA: {class_counts[1]}, ratio: {imbalance_ratio:.2f}x')

lr_class_weights = {i: len(y_train) / (len(class_counts) * c) for i, c in enumerate(class_counts)}
print(f'\nLR / sklearn class weights:')
print(f'  NORMAL={lr_class_weights[0]:.4f}, PNEUMONIA={lr_class_weights[1]:.4f}')

xgb_sample_weights = compute_sample_weight('balanced', y_train)
print(f'\nXGBoost per-sample weights:')
print(f'  NORMAL={xgb_sample_weights[y_train == 0][0]:.4f}, PNEUMONIA={xgb_sample_weights[y_train == 1][0]:.4f}')

per_class_weights = 1.0 / class_counts
per_sample_weights = per_class_weights[y_train]
print(f'\nCNN / ResNet sampler weights:')
print(f'  NORMAL={per_class_weights[0]:.5f}, PNEUMONIA={per_class_weights[1]:.5f}')

In [ ]:
lr_model = LogisticRegression(
    class_weight=lr_class_weights,
    max_iter=1000,
    C=1.0,
    solver='lbfgs',
    random_state=SEED
)
lr_model.fit(X_train, y_train)
print('Logistic Regression trained.')

In [ ]:
lr_val_proba = lr_model.predict_proba(X_val)[:, 1]
best_thresh_lr, best_f1_lr = 0.5, 0.0
for t in np.arange(0.3, 0.7, 0.01):
    score = f1_score(y_val, (lr_val_proba >= t).astype(int))
    if score > best_f1_lr:
        best_f1_lr, best_thresh_lr = score, t
print(f'Best val threshold: {best_thresh_lr:.2f}  (val F1={best_f1_lr:.4f})')

lr_test_proba = lr_model.predict_proba(X_test)[:, 1]
lr_test_pred = (lr_test_proba >= best_thresh_lr).astype(int)

print('\n--- Logistic Regression Test Results ---')
print(f'Accuracy: {accuracy_score(y_test, lr_test_pred):.4f}')
print(f'AUC-ROC: {roc_auc_score(y_test, lr_test_proba):.4f}')
print(f'F1: {f1_score(y_test, lr_test_pred):.4f}')
print()
print(classification_report(y_test, lr_test_pred, target_names=['NORMAL', 'PNEUMONIA']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, lr_test_pred, display_labels=['NORMAL', 'PNEUMONIA'],
    ax=axes[0], colorbar=False
)
axes[0].set_title('Logistic Regression - Confusion Matrix')

fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_test_proba)
axes[1].plot(fpr_lr, tpr_lr,
             label=f'AUC = {roc_auc_score(y_test, lr_test_proba):.4f}',
             color='steelblue')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Logistic Regression - ROC Curve')
axes[1].legend()
plt.tight_layout()
plt.show()

---
## Model 2: XGBoost

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=SEED,
    n_jobs=-1,
    device='cuda'
)
xgb_model.fit(
    X_train, y_train,
    sample_weight=xgb_sample_weights,
    eval_set=[(X_val, y_val)],
    verbose=50
)
print('XGBoost trained.')

In [ ]:
xgb_val_proba = xgb_model.predict_proba(X_val)[:, 1]
best_thresh_xgb, best_f1_xgb = 0.5, 0.0
for t in np.arange(0.3, 0.7, 0.01):
    score = f1_score(y_val, (xgb_val_proba >= t).astype(int))
    if score > best_f1_xgb:
        best_f1_xgb, best_thresh_xgb = score, t
print(f'Best val threshold: {best_thresh_xgb:.2f}  (val F1={best_f1_xgb:.4f})')

xgb_test_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_test_pred = (xgb_test_proba >= best_thresh_xgb).astype(int)

print('\n--- XGBoost Test Results ---')
print(f'Accuracy: {accuracy_score(y_test, xgb_test_pred):.4f}')
print(f'AUC-ROC: {roc_auc_score(y_test, xgb_test_proba):.4f}')
print(f'F1: {f1_score(y_test, xgb_test_pred):.4f}')
print()
print(classification_report(y_test, xgb_test_pred, target_names=['NORMAL', 'PNEUMONIA']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, xgb_test_pred, display_labels=['NORMAL', 'PNEUMONIA'],
    ax=axes[0], colorbar=False
)
axes[0].set_title('XGBoost - Confusion Matrix')

fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_test_proba)
axes[1].plot(fpr_xgb, tpr_xgb,
             label=f'AUC = {roc_auc_score(y_test, xgb_test_proba):.4f}',
             color='darkorange')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('XGBoost - ROC Curve')
axes[1].legend()
plt.tight_layout()
plt.show()

---
## Model 3: Custom CNN (PyTorch)

In [ ]:
class XRayDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = Image.open(self.file_paths[idx]).convert('L')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[idx], dtype=torch.float32)


CNN_IMG_SIZE = 64
BATCH_SIZE = 64
NUM_EPOCHS = 20

train_transform = transforms.Compose([
    transforms.Resize((CNN_IMG_SIZE, CNN_IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])
val_transform = transforms.Compose([
    transforms.Resize((CNN_IMG_SIZE, CNN_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

train_ds = XRayDataset(p_train, y_train, transform=train_transform)
val_ds = XRayDataset(p_val, y_val, transform=val_transform)
test_ds = XRayDataset(p_test, y_test, transform=val_transform)

sampler = WeightedRandomSampler(per_sample_weights, num_samples=len(y_train), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

In [ ]:
class ChestXRayCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)


cnn_model = ChestXRayCNN().to(DEVICE)
n_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(cnn_model)
print(f'\nTrainable parameters: {n_params:,}')

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [], 'val_acc': [],
    'train_auc': [], 'val_auc': []
}
best_val_auc, best_state = 0.0, None

for epoch in range(1, NUM_EPOCHS + 1):
    cnn_model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    t_proba, t_labels = [], []
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = cnn_model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        proba = torch.sigmoid(logits)
        t_loss += loss.item() * len(labels)
        t_correct += ((proba >= 0.5).long() == labels.long()).sum().item()
        t_total += len(labels)
        t_proba.extend(proba.detach().cpu().numpy())
        t_labels.extend(labels.detach().cpu().numpy())

    cnn_model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    v_proba, v_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = cnn_model(imgs)
            v_loss += criterion(logits, labels).item() * len(labels)
            proba = torch.sigmoid(logits)
            v_correct += ((proba >= 0.5).long() == labels.long()).sum().item()
            v_total += len(labels)
            v_proba.extend(proba.cpu().numpy())
            v_labels.extend(labels.cpu().numpy())

    t_loss /= t_total; v_loss /= v_total
    t_acc = t_correct / t_total
    v_acc = v_correct / v_total
    t_auc = roc_auc_score(t_labels, t_proba)
    v_auc = roc_auc_score(v_labels, v_proba)
    scheduler.step(v_auc)

    if v_auc > best_val_auc:
        best_val_auc = v_auc
        best_state = {k: v.clone() for k, v in cnn_model.state_dict().items()}
    history['train_loss'].append(t_loss); history['val_loss'].append(v_loss)
    history['train_acc'].append(t_acc); history['val_acc'].append(v_acc)
    history['train_auc'].append(t_auc); history['val_auc'].append(v_auc)

    print(f'Epoch {epoch:2d}/{NUM_EPOCHS}  '
          f'train_loss={t_loss:.4f}  train_acc={t_acc:.4f}  train_auc={t_auc:.4f}  '
          f'val_loss={v_loss:.4f}  val_acc={v_acc:.4f}  val_auc={v_auc:.4f}')

cnn_model.load_state_dict(best_state)
print(f'\nBest model restored (val AUC={best_val_auc:.4f})')

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(epochs_range, history['train_loss'], label='Train')
axes[0].plot(epochs_range, history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(epochs_range, history['train_acc'], label='Train')
axes[1].plot(epochs_range, history['val_acc'], label='Val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[2].plot(epochs_range, history['train_auc'], label='Train')
axes[2].plot(epochs_range, history['val_auc'], label='Val')
axes[2].set_title('AUC-ROC'); axes[2].set_xlabel('Epoch'); axes[2].legend()
plt.suptitle('CNN Training Curves', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
def collect_proba(loader, model, device):
    model.eval()
    proba_list, label_list = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            p = torch.sigmoid(model(imgs.to(device))).squeeze(-1).cpu().numpy()
            proba_list.extend(p.ravel())
            label_list.extend(labels.numpy().ravel())
    return np.array(proba_list), np.array(label_list)

cnn_val_proba, cnn_val_labels = collect_proba(val_loader, cnn_model, DEVICE)
cnn_test_proba, cnn_test_labels = collect_proba(test_loader, cnn_model, DEVICE)

best_thresh_cnn, best_f1_cnn = 0.5, 0.0
for t in np.arange(0.3, 0.7, 0.01):
    score = f1_score(cnn_val_labels, (cnn_val_proba >= t).astype(int))
    if score > best_f1_cnn:
        best_f1_cnn, best_thresh_cnn = score, t
print(f'Best val threshold: {best_thresh_cnn:.2f}  (val F1={best_f1_cnn:.4f})')

cnn_test_pred = (cnn_test_proba >= best_thresh_cnn).astype(int)

print('\n--- CNN Test Results ---')
print(f'Accuracy: {accuracy_score(cnn_test_labels, cnn_test_pred):.4f}')
print(f'AUC-ROC: {roc_auc_score(cnn_test_labels, cnn_test_proba):.4f}')
print(f'F1: {f1_score(cnn_test_labels, cnn_test_pred):.4f}')
print()
print(classification_report(cnn_test_labels, cnn_test_pred, target_names=['NORMAL', 'PNEUMONIA']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(
    cnn_test_labels, cnn_test_pred, display_labels=['NORMAL', 'PNEUMONIA'],
    ax=axes[0], colorbar=False
)
axes[0].set_title('CNN - Confusion Matrix')

fpr_cnn, tpr_cnn, _ = roc_curve(cnn_test_labels, cnn_test_proba)
axes[1].plot(fpr_cnn, tpr_cnn,
             label=f'AUC = {roc_auc_score(cnn_test_labels, cnn_test_proba):.4f}',
             color='green')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('CNN - ROC Curve')
axes[1].legend()
plt.tight_layout()
plt.show()

---
## Model 4: ResNet18 (Transfer Learning)

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

RN_IMG_SIZE = 224
RN_BATCH = 32
RN_EPOCHS = 20

rn_train_transform = transforms.Compose([
    transforms.Resize((RN_IMG_SIZE, RN_IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
rn_val_transform = transforms.Compose([
    transforms.Resize((RN_IMG_SIZE, RN_IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

rn_train_ds = XRayDataset(p_train, y_train, transform=rn_train_transform)
rn_val_ds = XRayDataset(p_val, y_val, transform=rn_val_transform)
rn_test_ds = XRayDataset(p_test, y_test, transform=rn_val_transform)

rn_sampler = WeightedRandomSampler(per_sample_weights, num_samples=len(y_train), replacement=True)

rn_train_loader = DataLoader(rn_train_ds, batch_size=RN_BATCH, sampler=rn_sampler, num_workers=0)
rn_val_loader = DataLoader(rn_val_ds, batch_size=RN_BATCH, shuffle=False, num_workers=0)
rn_test_loader = DataLoader(rn_test_ds, batch_size=RN_BATCH, shuffle=False, num_workers=0)

print(f'Train batches: {len(rn_train_loader)}, Val: {len(rn_val_loader)}, Test: {len(rn_test_loader)}')

In [ ]:
rn_model = resnet18(weights=ResNet18_Weights.DEFAULT)
rn_model.fc = nn.Linear(rn_model.fc.in_features, 1)
rn_model = rn_model.to(DEVICE)

WARMUP_EPOCHS = 5
FINETUNE_EPOCHS = RN_EPOCHS - WARMUP_EPOCHS

rn_criterion = nn.BCEWithLogitsLoss()

n_params = sum(p.numel() for p in rn_model.parameters() if p.requires_grad)
print(f'ResNet18 total trainable params: {n_params:,}')
print(f'FC layer: {rn_model.fc}')

In [ ]:
def run_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_proba, all_labels = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            logits = model(imgs).squeeze(1)
            loss = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            proba = torch.sigmoid(logits)
            correct += ((proba >= 0.5).long() == labels.long()).sum().item()
            total += len(labels)
            all_proba.extend(proba.detach().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, np.array(all_proba), np.array(all_labels)


rn_history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [], 'val_acc': [],
    'train_auc': [], 'val_auc': []
}
rn_best_auc, rn_best_state = 0.0, None

for param in rn_model.parameters():
    param.requires_grad = False
rn_model.fc.weight.requires_grad = True
rn_model.fc.bias.requires_grad = True

rn_optimizer = optim.Adam(rn_model.fc.parameters(), lr=1e-3)
print(f'Phase 1 — warm-up ({WARMUP_EPOCHS} epochs, head only)')

for epoch in range(1, WARMUP_EPOCHS + 1):
    t_loss, t_acc, t_proba, t_lbl = run_epoch(rn_model, rn_train_loader, rn_criterion, rn_optimizer, DEVICE, train=True)
    v_loss, v_acc, v_proba, v_lbl = run_epoch(rn_model, rn_val_loader, rn_criterion, rn_optimizer, DEVICE, train=False)
    t_auc = roc_auc_score(t_lbl, t_proba)
    v_auc = roc_auc_score(v_lbl, v_proba)
    if v_auc > rn_best_auc:
        rn_best_auc = v_auc
        rn_best_state = {k: v.clone() for k, v in rn_model.state_dict().items()}
    rn_history['train_loss'].append(t_loss); rn_history['val_loss'].append(v_loss)
    rn_history['train_acc'].append(t_acc); rn_history['val_acc'].append(v_acc)
    rn_history['train_auc'].append(t_auc); rn_history['val_auc'].append(v_auc)
    print(f'  Epoch {epoch:2d}/{WARMUP_EPOCHS}  train_loss={t_loss:.4f}  train_acc={t_acc:.4f}  train_auc={t_auc:.4f}  '
          f'val_loss={v_loss:.4f}  val_acc={v_acc:.4f}  val_auc={v_auc:.4f}')

for param in rn_model.parameters():
    param.requires_grad = True

rn_optimizer = optim.Adam([
    {'params': [p for n, p in rn_model.named_parameters() if 'fc' not in n], 'lr': 1e-4},
    {'params': rn_model.fc.parameters(), 'lr': 1e-3}
])
rn_scheduler = optim.lr_scheduler.ReduceLROnPlateau(rn_optimizer, mode='max', factor=0.5, patience=3)

print(f'\nPhase 2 — fine-tune all ({FINETUNE_EPOCHS} epochs)')
for epoch in range(WARMUP_EPOCHS + 1, RN_EPOCHS + 1):
    t_loss, t_acc, t_proba, t_lbl = run_epoch(rn_model, rn_train_loader, rn_criterion, rn_optimizer, DEVICE, train=True)
    v_loss, v_acc, v_proba, v_lbl = run_epoch(rn_model, rn_val_loader, rn_criterion, rn_optimizer, DEVICE, train=False)
    t_auc = roc_auc_score(t_lbl, t_proba)
    v_auc = roc_auc_score(v_lbl, v_proba)
    rn_scheduler.step(v_auc)
    if v_auc > rn_best_auc:
        rn_best_auc = v_auc
        rn_best_state = {k: v.clone() for k, v in rn_model.state_dict().items()}
    rn_history['train_loss'].append(t_loss); rn_history['val_loss'].append(v_loss)
    rn_history['train_acc'].append(t_acc); rn_history['val_acc'].append(v_acc)
    rn_history['train_auc'].append(t_auc); rn_history['val_auc'].append(v_auc)
    print(f'  Epoch {epoch:2d}/{RN_EPOCHS}  train_loss={t_loss:.4f}  train_acc={t_acc:.4f}  train_auc={t_auc:.4f}  '
          f'val_loss={v_loss:.4f}  val_acc={v_acc:.4f}  val_auc={v_auc:.4f}')

rn_model.load_state_dict(rn_best_state)
print(f'\nBest model restored (val AUC={rn_best_auc:.4f})')

In [ ]:
rn_epochs_range = range(1, len(rn_history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(rn_epochs_range, rn_history['train_loss'], label='Train')
axes[0].plot(rn_epochs_range, rn_history['val_loss'], label='Val')
axes[0].axvline(WARMUP_EPOCHS + 0.5, color='gray', linestyle='--', label='Unfreeze backbone')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(rn_epochs_range, rn_history['train_acc'], label='Train')
axes[1].plot(rn_epochs_range, rn_history['val_acc'], label='Val')
axes[1].axvline(WARMUP_EPOCHS + 0.5, color='gray', linestyle='--', label='Unfreeze backbone')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[2].plot(rn_epochs_range, rn_history['train_auc'], label='Train')
axes[2].plot(rn_epochs_range, rn_history['val_auc'], label='Val')
axes[2].axvline(WARMUP_EPOCHS + 0.5, color='gray', linestyle='--', label='Unfreeze backbone')
axes[2].set_title('AUC-ROC'); axes[2].set_xlabel('Epoch'); axes[2].legend()
plt.suptitle('ResNet18 Training Curves', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
rn_val_proba, rn_val_labels = collect_proba(rn_val_loader, rn_model, DEVICE)
rn_test_proba, rn_test_labels = collect_proba(rn_test_loader, rn_model, DEVICE)

best_thresh_rn, best_f1_rn = 0.5, 0.0
for t in np.arange(0.3, 0.7, 0.01):
    score = f1_score(rn_val_labels, (rn_val_proba >= t).astype(int))
    if score > best_f1_rn:
        best_f1_rn, best_thresh_rn = score, t
print(f'Best val threshold: {best_thresh_rn:.2f}  (val F1={best_f1_rn:.4f})')

rn_test_pred = (rn_test_proba >= best_thresh_rn).astype(int)

print('\n--- ResNet18 Test Results ---')
print(f'Accuracy: {accuracy_score(rn_test_labels, rn_test_pred):.4f}')
print(f'AUC-ROC: {roc_auc_score(rn_test_labels, rn_test_proba):.4f}')
print(f'F1: {f1_score(rn_test_labels, rn_test_pred):.4f}')
print()
print(classification_report(rn_test_labels, rn_test_pred, target_names=['NORMAL', 'PNEUMONIA']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(
    rn_test_labels, rn_test_pred, display_labels=['NORMAL', 'PNEUMONIA'],
    ax=axes[0], colorbar=False
)
axes[0].set_title('ResNet18 - Confusion Matrix')

fpr_rn, tpr_rn, _ = roc_curve(rn_test_labels, rn_test_proba)
axes[1].plot(fpr_rn, tpr_rn,
             label=f'AUC = {roc_auc_score(rn_test_labels, rn_test_proba):.4f}',
             color='purple')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ResNet18 - ROC Curve')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
correct_mask = rn_test_pred == rn_test_labels.astype(int)
incorrect_mask = ~correct_mask

correct_idxs = np.where(correct_mask)[0][:6]
incorrect_idxs = np.where(incorrect_mask)[0][:6]

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for col, idx in enumerate(correct_idxs):
    img = Image.open(p_test[idx]).convert('L').resize((IMG_SIZE, IMG_SIZE))
    axes[0, col].imshow(np.array(img), cmap='gray')
    axes[0, col].axis('off')
    if col == 0:
        axes[0, col].set_title('Correct', fontsize=11, loc='left')
for col, idx in enumerate(incorrect_idxs):
    img = Image.open(p_test[idx]).convert('L').resize((IMG_SIZE, IMG_SIZE))
    axes[1, col].imshow(np.array(img), cmap='gray')
    axes[1, col].axis('off')
    if col == 0:
        axes[1, col].set_title('Incorrect', fontsize=11, loc='left')
plt.suptitle('ResNet18 — Correct vs Incorrect Predictions', fontsize=13)
plt.tight_layout()
plt.show()

---
## Results Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost', 'CNN', 'ResNet18'],
    'Accuracy': [
        accuracy_score(y_test, lr_test_pred),
        accuracy_score(y_test, xgb_test_pred),
        accuracy_score(cnn_test_labels, cnn_test_pred),
        accuracy_score(rn_test_labels, rn_test_pred)
    ],
    'AUC-ROC': [
        roc_auc_score(y_test, lr_test_proba),
        roc_auc_score(y_test, xgb_test_proba),
        roc_auc_score(cnn_test_labels, cnn_test_proba),
        roc_auc_score(rn_test_labels, rn_test_proba)
    ],
    'F1 (PNEUMONIA)': [
        f1_score(y_test, lr_test_pred),
        f1_score(y_test, xgb_test_pred),
        f1_score(cnn_test_labels, cnn_test_pred),
        f1_score(rn_test_labels, rn_test_pred)
    ]
}).set_index('Model').round(4)
display(results)

In [ ]:
plt.figure(figsize=(8, 5))
for fpr, tpr, proba, labels, label, color in [
    (fpr_lr, tpr_lr, lr_test_proba, y_test, 'Logistic Regression', 'steelblue'),
    (fpr_xgb, tpr_xgb, xgb_test_proba, y_test, 'XGBoost', 'darkorange'),
    (fpr_cnn, tpr_cnn, cnn_test_proba, cnn_test_labels, 'CNN', 'green'),
    (fpr_rn, tpr_rn, rn_test_proba, rn_test_labels, 'ResNet18', 'purple'),
]:
    auc = roc_auc_score(labels, proba)
    plt.plot(fpr, tpr, label=f'{label}  (AUC={auc:.4f})', color=color)
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison - All Models')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
model_names = ['Logistic Regression', 'XGBoost', 'CNN', 'ResNet18']
colors = ['steelblue', 'darkorange', 'green', 'purple']
metrics = ['Accuracy', 'AUC-ROC', 'F1']
values = [
    [accuracy_score(y_test, lr_test_pred), roc_auc_score(y_test, lr_test_proba), f1_score(y_test, lr_test_pred)],
    [accuracy_score(y_test, xgb_test_pred), roc_auc_score(y_test, xgb_test_proba), f1_score(y_test, xgb_test_pred)],
    [accuracy_score(cnn_test_labels, cnn_test_pred), roc_auc_score(cnn_test_labels, cnn_test_proba), f1_score(cnn_test_labels, cnn_test_pred)],
    [accuracy_score(rn_test_labels, rn_test_pred), roc_auc_score(rn_test_labels, rn_test_proba), f1_score(rn_test_labels, rn_test_pred)],
]

x = np.arange(len(metrics))
width = 0.18
fig, ax = plt.subplots(figsize=(10, 5))
for i, (name, vals, color) in enumerate(zip(model_names, values, colors)):
    ax.bar(x + i * width, vals, width, label=name, color=color)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics)
ax.set_ylim(0.88, 1.01)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (y_true, y_pred, title) in zip(axes, [
    (y_test, lr_test_pred, 'Logistic Regression'),
    (y_test, xgb_test_pred, 'XGBoost'),
    (cnn_test_labels, cnn_test_pred, 'CNN'),
    (rn_test_labels, rn_test_pred, 'ResNet18'),
]):
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred, display_labels=['NORMAL', 'PNEUMONIA'], ax=ax, colorbar=False)
    ax.set_title(title)
plt.suptitle('Confusion Matrices — All Models', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, ax = plt.subplots(figsize=(7, 5))
for proba, labels, label, color in [
    (lr_test_proba, y_test, 'Logistic Regression', 'steelblue'),
    (xgb_test_proba, y_test, 'XGBoost', 'darkorange'),
    (cnn_test_proba, cnn_test_labels, 'CNN', 'green'),
    (rn_test_proba, rn_test_labels, 'ResNet18', 'purple'),
]:
    prec, rec, _ = precision_recall_curve(labels, proba)
    ap = average_precision_score(labels, proba)
    ax.plot(rec, prec, label=f'{label}  (AP={ap:.4f})', color=color)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves — All Models')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.01)
fig, ax = plt.subplots(figsize=(9, 5))
for proba, labels, label, color in [
    (lr_test_proba, y_test, 'Logistic Regression', 'steelblue'),
    (xgb_test_proba, y_test, 'XGBoost', 'darkorange'),
    (cnn_test_proba, cnn_test_labels, 'CNN', 'green'),
    (rn_test_proba, rn_test_labels, 'ResNet18', 'purple'),
]:
    f1s = [f1_score(labels, (proba >= t).astype(int)) for t in thresholds]
    ax.plot(thresholds, f1s, label=label, color=color)
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('F1 Score')
ax.set_title('F1 Score vs Decision Threshold — All Models')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, (proba, labels, title) in zip(axes, [
    (lr_test_proba, y_test, 'Logistic Regression'),
    (xgb_test_proba, y_test, 'XGBoost'),
    (cnn_test_proba, cnn_test_labels, 'CNN'),
    (rn_test_proba, rn_test_labels, 'ResNet18'),
]):
    p = np.asarray(proba).ravel()
    l = np.asarray(labels).ravel()
    ax.violinplot([p[l == 0], p[l == 1]], positions=[0, 1], showmedians=True)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['NORMAL', 'PNEUMONIA'])
    ax.set_ylabel('Predicted Probability')
    ax.set_ylim(0, 1)
    ax.set_title(title)
plt.suptitle('Prediction Probability Distribution by True Label', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (proba, title) in zip(axes.flat, [
    (lr_test_proba, 'Logistic Regression'),
    (xgb_test_proba, 'XGBoost'),
    (cnn_test_proba, 'CNN'),
    (rn_test_proba, 'ResNet18'),
]):
    ax.hist(proba, bins=40, color='steelblue', edgecolor='white')
    ax.set_xlabel('Predicted Probability (PNEUMONIA)')
    ax.set_ylabel('Count')
    ax.set_title(title)
plt.suptitle('Probability Histogram — All Models', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)
rn_epochs_range = range(1, len(rn_history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18,4))

# ---- Loss Plot ----
axes[0].plot(epochs_range, history['train_loss'], label='CNN Train')
axes[0].plot(epochs_range, history['val_loss'], label='CNN Val')

axes[0].plot(rn_epochs_range, rn_history['train_loss'], linestyle='--', label='ResNet Train')
axes[0].plot(rn_epochs_range, rn_history['val_loss'], linestyle='--', label='ResNet Val')

axes[0].set_title('Model Loss Comparison')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()


# ---- Accuracy Plot ----
axes[1].plot(epochs_range, history['train_acc'], label='CNN Train')
axes[1].plot(epochs_range, history['val_acc'], label='CNN Val')

axes[1].plot(rn_epochs_range, rn_history['train_acc'], linestyle='--', label='ResNet Train')
axes[1].plot(rn_epochs_range, rn_history['val_acc'], linestyle='--', label='ResNet Val')

axes[1].set_title('Model Accuracy Comparison')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

# ---- AUC Plot ----
axes[2].plot(epochs_range, history['train_auc'], label='CNN Train')
axes[2].plot(epochs_range, history['val_auc'], label='CNN Val')

axes[2].plot(rn_epochs_range, rn_history['train_auc'], linestyle='--', label='ResNet Train')
axes[2].plot(rn_epochs_range, rn_history['val_auc'], linestyle='--', label='ResNet Val')

axes[2].set_title('Model AUC Comparison')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('AUC-ROC')
axes[2].legend()

plt.suptitle('CNN vs ResNet18 Training Curves', fontsize=13)
plt.tight_layout()
plt.show()

### CNN Ablation Study

This section folds the standalone `cnn_ablation.py` workflow into the notebook. It reuses the existing split variables (`p_train`, `y_train`, `p_val`, `y_val`, `p_test`, `y_test`) and cached results if they already exist, so duplicate training work is skipped.


In [ ]:
from dataclasses import asdict, dataclass
import csv
import json
import random

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

@dataclass
class AblationResult:
    variant: str
    use_batch_norm: bool
    use_augmentation: bool
    threshold: float
    accuracy: float
    auc_roc: float
    recall_pneumonia: float
    f1_pneumonia: float

class ChestXRayAblationCNN(nn.Module):
    def __init__(self, use_batch_norm: bool):
        super().__init__()

        def block(in_channels, out_channels):
            layers = [nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)]
            if use_batch_norm:
                layers.append(nn.BatchNorm2d(out_channels))
            layers.extend([nn.ReLU(inplace=True), nn.MaxPool2d(2)])
            return layers

        self.features = nn.Sequential(
            *block(1, 32),
            *block(32, 64),
            *block(64, 128),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 1),
        )

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)

def make_cnn_transform(use_augmentation: bool):
    items = [transforms.Resize((CNN_IMG_SIZE, CNN_IMG_SIZE))]
    if use_augmentation:
        items.extend([transforms.RandomHorizontalFlip(), transforms.RandomRotation(10)])
    items.extend([transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
    return transforms.Compose(items)

def run_cnn_ablation_variant(variant_name, use_batch_norm, use_augmentation, epochs=3):
    set_seed(SEED)
    train_transform = make_cnn_transform(use_augmentation)
    eval_transform = make_cnn_transform(False)

    train_ds = XRayDataset(p_train, y_train, transform=train_transform)
    val_ds = XRayDataset(p_val, y_val, transform=eval_transform)
    test_ds = XRayDataset(p_test, y_test, transform=eval_transform)

    sampler = WeightedRandomSampler(
        torch.DoubleTensor(per_sample_weights),
        num_samples=len(y_train),
        replacement=True,
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = ChestXRayAblationCNN(use_batch_norm=use_batch_norm).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

    best_val_auc, best_state = 0.0, None
    for epoch in range(1, epochs + 1):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

        val_proba, val_labels = collect_proba(val_loader, model, DEVICE)
        val_auc = roc_auc_score(val_labels, val_proba)
        scheduler.step(val_auc)
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"{variant_name} | epoch {epoch:2d}/{epochs} | val_auc={val_auc:.4f}")

    model.load_state_dict(best_state)
    val_proba, val_labels = collect_proba(val_loader, model, DEVICE)
    best_threshold, best_val_f1 = 0.5, -1.0
    for threshold in np.arange(0.3, 0.71, 0.01):
        preds = (val_proba >= threshold).astype(int)
        score = f1_score(val_labels, preds)
        if score > best_val_f1:
            best_val_f1 = score
            best_threshold = float(round(threshold, 2))

    test_proba, test_labels = collect_proba(test_loader, model, DEVICE)
    test_pred = (test_proba >= best_threshold).astype(int)

    return AblationResult(
        variant=variant_name,
        use_batch_norm=use_batch_norm,
        use_augmentation=use_augmentation,
        threshold=best_threshold,
        accuracy=float((test_pred == test_labels).mean()),
        auc_roc=float(roc_auc_score(test_labels, test_proba)),
        recall_pneumonia=float(recall_score(test_labels, test_pred, pos_label=1)),
        f1_pneumonia=float(f1_score(test_labels, test_pred, pos_label=1)),
    )


In [ ]:
ABLATION_RESULTS_CSV = 'cnn_ablation_results.csv'
ABLATION_RESULTS_JSON = 'cnn_ablation_results.json'
CNN_ABLATION_EPOCHS = 3

if os.path.exists(ABLATION_RESULTS_CSV):
    cnn_ablation_results = pd.read_csv(ABLATION_RESULTS_CSV)
else:
    variants = [
        ('Without BN, Without Aug', False, False),
        ('Without BN, With Aug', False, True),
        ('With BN, No Aug', True, False),
        ('With BN, With Aug', True, True),
    ]
    rows = []
    for variant_name, use_batch_norm, use_augmentation in variants:
        rows.append(asdict(run_cnn_ablation_variant(variant_name, use_batch_norm, use_augmentation, epochs=CNN_ABLATION_EPOCHS)))
    cnn_ablation_results = pd.DataFrame(rows)
    cnn_ablation_results.to_csv(ABLATION_RESULTS_CSV, index=False)
    cnn_ablation_results.to_json(ABLATION_RESULTS_JSON, orient='records', indent=2)

cnn_ablation_results


In [ ]:
cnn_ablation_table = cnn_ablation_results[[
    'variant', 'auc_roc', 'recall_pneumonia', 'f1_pneumonia'
]].rename(columns={
    'auc_roc': 'CNN AUC-ROC',
    'recall_pneumonia': 'CNN Recall (PNEUMONIA)',
    'f1_pneumonia': 'CNN F1 Score (PNEUMONIA)'
}).round(4)
display(cnn_ablation_table)
